# Topic 5: Interview Masterclass

> **Phase 7 → Week 12 → Topic 5**

---

## How Spark Interviews Work

```
Round 1: Conceptual (phone screen)
  "What is lazy evaluation?" "Explain shuffles."
  "Difference between repartition and coalesce?"
  → Memorize the cheat sheets from all 12 weeks.

Round 2: Coding (live coding or take-home)
  Write PySpark to: join, aggregate, window, handle nulls, deduplicate.
  → Practice writing from scratch without autocomplete.

Round 3: System Design
  "Design a real-time fraud detection pipeline."
  "How would you process 10 TB of daily logs?"
  → Apply Phase 5-6 patterns: Medallion, Streaming, exactly-once.

Round 4: Performance & Debugging
  "Why is this job slow?" "Spark UI screenshot — what's wrong?"
  "How do you fix data skew?"
  → Know OOM causes, skew fixes, AQE, caching strategy.
```

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql import Window

spark = SparkSession.builder \
    .appName("Week12 - Interview Masterclass") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Interview practice environment ready")

## Part 1: Top 20 Interview Questions with Answers

In [ ]:
print("""
TOP 20 SPARK INTERVIEW QUESTIONS:
════════════════════════════════════════════════════════════════

Q1. What is lazy evaluation in Spark?
  Transformations (map, filter, groupBy) build a DAG — no computation happens.
  Only when an Action (count, collect, write) is called does Spark execute the plan.
  Benefit: Catalyst can optimize the full DAG before executing anything.

Q2. What is the difference between repartition and coalesce?
  repartition(N): full shuffle, can increase OR decrease partitions, even distribution.
  coalesce(N):    no shuffle (usually), can only DECREASE partitions, may be uneven.
  Use coalesce to reduce file count before writing. Use repartition to fix skew.

Q3. When does a shuffle occur?
  Any wide transformation: groupBy, join, distinct, repartition, orderBy.
  Data moves between executors → expensive (network + disk I/O).
  Narrow: filter, map, withColumn — no shuffle.

Q4. What is the difference between cache() and persist()?
  cache() = persist(StorageLevel.MEMORY_AND_DISK)  (convenience method).
  persist() lets you choose: MEMORY_ONLY, DISK_ONLY, MEMORY_AND_DISK, MEMORY_ONLY_SER.
  Use MEMORY_AND_DISK for safety (spill to disk if memory full).
  Always unpersist() when done — releases executor memory.

Q5. What is data skew and how do you fix it?
  Skew: one key has far more data than others → one task runs 100× longer.
  Detection: Spark UI Stage tasks → max task time >> median task time.
  Fixes: AQE skew join (auto), salting (add random suffix to key + replicate small side),
         isolate skewed key with a separate broadcast join.

Q6. What is AQE (Adaptive Query Execution)?
  Spark 3.0+ feature: re-optimizes the query plan at runtime using actual shuffle statistics.
  Three main features:
    a) Coalesce shuffle partitions: merges empty/tiny partitions → fewer tasks
    b) Switch join strategy: changes SMJ to BHJ if one side turns out small
    c) Skew join: splits large partitions into smaller ones
  Enable: spark.sql.adaptive.enabled=true (default in Spark 3.2+)

Q7. What is the difference between DataFrame and RDD?
  RDD: low-level, untyped distributed collection, no optimizer, manual partitioning.
  DataFrame: typed rows with named columns, Catalyst + Tungsten optimization,
             SQL-like API, 10-100× faster than equivalent RDD code.
  Use DataFrame/Dataset. Use RDD only for legacy code or operations with no DF equivalent.

Q8. What is Broadcast Join and when is it used?
  Spark broadcasts the small table to all executors → local hash join, no shuffle.
  Triggered automatically when small table < autoBroadcastJoinThreshold (10 MB default).
  Force with F.broadcast(df) or hint('broadcast').
  Always use for dimension table joins — massive speedup.

Q9. What is checkpointing in streaming?
  Durable log (S3/HDFS) that records: offsets read, batches committed, state store.
  Enables exactly-once processing: restart resumes from the last committed offset.
  Required for all production streaming jobs.
  .option("checkpointLocation", "s3://bucket/checkpoint/job1")

Q10. What is a watermark in Structured Streaming?
  Threshold for late data: withWatermark("event_time", "15 minutes").
  Watermark = max(event_time seen) - 15 min.
  Events older than watermark are dropped.
  Enables bounded state store (old keys evicted).
  Required for stream-stream joins and memory-safe stateful aggregations.
════════════════════════════════════════════════════════════════
""")

In [ ]:
print("""
Q11. Explain Delta Lake transaction log.
  _delta_log directory: one JSON file per transaction.
  Each entry records: files added, files removed.
  Spark reconstructs current state by replaying log from latest checkpoint.
  Enables: ACID (one atomic log entry), time travel (replay to version N),
           MERGE upsert, schema enforcement, schema evolution.

Q12. What is VACUUM in Delta Lake?
  Deletes old Parquet files no longer referenced in the Delta log within retention period.
  Default: 7 days. After VACUUM, cannot time-travel before the retention boundary.
  Run regularly in production to control S3/HDFS costs.

Q13. What is the Medallion Architecture?
  Bronze: raw data, append-only, schema-on-read, never transform.
  Silver: cleaned + validated, schema enforced, deduplicated, joined with dims.
  Gold: aggregated, business-ready, optimized for specific use cases (BI, ML).
  Each layer reprocessable from the previous. Bronze = source of truth.

Q14. What is SCD Type 2?
  Slowly Changing Dimension that preserves full history.
  When a record changes: close old row (set end_date, is_current=False),
  insert new row (start_date=today, end_date=null, is_current=True).
  Requires: surrogate key (not natural key), start_date, end_date, is_current.

Q15. What is the difference between exactly-once and at-least-once?
  At-least-once: no data loss, but duplicates possible on retry.
  Exactly-once: no loss, no duplicates. Requires: replayable source +
                checkpoint + idempotent sink.
  Spark Structured Streaming achieves exactly-once with file/Delta sinks.
  Kafka sink is at-least-once.

Q16. What is the difference between tumbling and sliding windows?
  Tumbling: fixed-size, non-overlapping. Each event in exactly 1 window.
  window("ts", "10 minutes") → windows: 0-10, 10-20, 20-30, ...
  Sliding: fixed-size, overlapping. Each event in size/slide windows.
  window("ts", "10 minutes", "5 minutes") → event at 10:07 in windows 10:00-10:10 and 10:05-10:15.

Q17. How do you handle OOM in Spark?
  a) Increase shuffle partitions (smaller partitions per task)
  b) Fix data skew (salt, AQE)
  c) Unpersist cached DataFrames when done
  d) Use broadcast join to avoid large shuffles
  e) Increase memoryOverhead for Python UDFs
  f) Switch to RocksDB state store for large streaming state

Q18. What is the Catalyst Optimizer?
  Spark's query optimizer: parses SQL/DataFrame → logical plan →
  optimized logical plan (predicate pushdown, constant folding, etc.) →
  physical plans (multiple strategies) → cost-based best plan.
  Works for both SQL and DataFrame API (same optimizer).

Q19. What is a Pandas UDF and when should you use it?
  Vectorized UDF using Apache Arrow for batch serialization.
  Receives pandas Series, returns pandas Series. 2-10× faster than regular UDFs.
  Use when: you need Python/pandas/scikit-learn/NumPy in a transformation.
  Prefer built-in F functions where possible (zero Python overhead).

Q20. How do you make a Spark ETL pipeline idempotent?
  Idempotent = same result whether run once or N times.
  Techniques: (a) overwrite + replaceWhere for date partitions,
  (b) Delta MERGE with batch_id/run_id as dedup key,
  (c) write to a partition path that is fully overwritten each run,
  (d) trigger(once=True) + file source checkpoint.
════════════════════════════════════════════════════════════════
""")

## Part 2: Live Coding Practice

In [ ]:
# Practice dataset: orders, customers, products
orders = spark.createDataFrame([
    ("O001", "C1", "P10", 150.0, "2024-01-15", "delivered"),
    ("O002", "C2", "P20",  89.5, "2024-01-15", "pending"),
    ("O003", "C1", "P10", 500.0, "2024-01-16", "delivered"),
    ("O004", "C3", "P30",  45.0, "2024-01-16", "cancelled"),
    ("O005", "C2", "P20", 300.0, "2024-01-17", "delivered"),
    ("O006", "C1", "P30",  75.0, "2024-01-17", "pending"),
    ("O007", "C3", "P10", 220.0, "2024-01-18", "delivered"),
    ("O008", "C1", "P20", 180.0, "2024-01-18", "delivered"),
], ["order_id", "customer_id", "product_id", "amount", "order_date", "status"])

customers = spark.createDataFrame([
    ("C1", "Alice",   "Mumbai",    "Gold"),
    ("C2", "Bob",     "Delhi",     "Silver"),
    ("C3", "Charlie", "Bangalore", "Bronze"),
], ["customer_id", "name", "city", "tier"])

products = spark.createDataFrame([
    ("P10", "Laptop",   "electronics", 0.18),
    ("P20", "T-Shirt",  "clothing",    0.12),
    ("P30", "Headphones", "electronics", 0.18),
], ["product_id", "product_name", "category", "tax_rate"])

print("Practice DataFrames ready:")
print(f"  orders: {orders.count()} rows")
print(f"  customers: {customers.count()} rows")
print(f"  products: {products.count()} rows")

In [ ]:
# CODING CHALLENGE 1: Revenue per customer with tier and product
# JOIN orders → customers → products
# Compute: total_revenue, order_count, avg_order_value per customer
# Sort by total_revenue descending

result1 = orders \
    .join(F.broadcast(customers), "customer_id") \
    .join(F.broadcast(products), "product_id") \
    .filter(F.col("status") != "cancelled") \
    .groupBy("customer_id", "name", "city", "tier") \
    .agg(
        F.count("*").alias("order_count"),
        F.round(F.sum("amount"), 2).alias("total_revenue"),
        F.round(F.avg("amount"), 2).alias("avg_order"),
    ) \
    .orderBy(F.col("total_revenue").desc())

print("Challenge 1: Revenue per customer")
result1.show()

In [ ]:
# CODING CHALLENGE 2: Running total per customer by date (Window function)
w = Window.partitionBy("customer_id").orderBy("order_date") \
           .rowsBetween(Window.unboundedPreceding, Window.currentRow)

result2 = orders \
    .filter(F.col("status") != "cancelled") \
    .withColumn("running_total", F.round(F.sum("amount").over(w), 2)) \
    .withColumn("order_rank",    F.rank().over(Window.partitionBy("customer_id").orderBy(F.col("amount").desc()))) \
    .select("customer_id", "order_id", "order_date", "amount", "running_total", "order_rank")

print("Challenge 2: Running total + rank per customer")
result2.orderBy("customer_id", "order_date").show()

In [ ]:
# CODING CHALLENGE 3: Top product per category by revenue
# Pattern: groupBy → window rank → filter rank=1

with_product = orders \
    .filter(F.col("status") == "delivered") \
    .join(F.broadcast(products), "product_id")

product_revenue = with_product \
    .groupBy("product_id", "product_name", "category") \
    .agg(F.round(F.sum("amount"), 2).alias("revenue"))

w_cat = Window.partitionBy("category").orderBy(F.col("revenue").desc())

result3 = product_revenue \
    .withColumn("rank", F.rank().over(w_cat)) \
    .filter(F.col("rank") == 1) \
    .drop("rank")

print("Challenge 3: Top product per category by revenue")
result3.show()

In [ ]:
# CODING CHALLENGE 4: Deduplication — keep latest order per customer per product
# Pattern: row_number() window partitioned by key, ordered by latest date

dup_orders = orders.union(spark.createDataFrame([
    ("O001", "C1", "P10", 155.0, "2024-01-20", "updated"),  # duplicate O001 with new amount
    ("O003", "C1", "P10", 505.0, "2024-01-20", "updated"),  # duplicate O003
], ["order_id", "customer_id", "product_id", "amount", "order_date", "status"]))

w_dedup = Window.partitionBy("order_id").orderBy(F.col("order_date").desc())

deduped = dup_orders \
    .withColumn("_rn", F.row_number().over(w_dedup)) \
    .filter(F.col("_rn") == 1) \
    .drop("_rn")

print(f"Before dedup: {dup_orders.count()} rows")
print(f"After dedup:  {deduped.count()} rows (should be {orders.count()})")
print("Latest version of each order:")
deduped.filter(F.col("order_id").isin("O001", "O003")).show()

## Part 3: System Design — Sample Answer

In [ ]:
print("""
SYSTEM DESIGN: Real-time Order Analytics Platform
════════════════════════════════════════════════════════════════

Requirements:
  - 100K orders/minute from 10 regional apps
  - Real-time dashboard: revenue per category (1-min lag)
  - Daily batch report: SCD2 customer table, top products
  - Historical queries: any day in the last 3 years
  - SLA: dashboard < 2 minutes lag, batch < 2 hours

Architecture:

  1. INGESTION:
     10 apps → Kafka (10 partitions × 10K orders/min per partition)
     Each order: {order_id, customer_id, amount, category, region, event_time}

  2. STREAMING LAYER (Spark Structured Streaming):
     Source: Kafka topic "orders" (all regions)
     Transform:
       a) Parse JSON payload (from_json with ORDER_SCHEMA)
       b) Validate (dead letter for bad records)
       c) Enrich: stream-static join with static product catalog
       d) Window agg: 1-min tumbling window, count + revenue per category
          withWatermark("event_time", "5 minutes")
          groupBy(window("event_time", "1 minute"), "category")
     Output: 
       - Window agg → Delta table (dashboard reads this)
       - Raw events → Bronze Delta table (append-only)
     Checkpoint: S3 (for exactly-once)
     Trigger: processingTime="30 seconds"

  3. BATCH LAYER (Spark on EMR/Databricks, nightly):
     Bronze → Silver: validate schema, deduplicate (row_number by event_time),
                      fix nulls, standardize categories
     Silver → Gold:
       - SCD2 customer table (MERGE with open/close logic)
       - Daily revenue by product/region/category
       - Cohort analysis (first purchase date → retention)
     Tools: Delta MERGE, OPTIMIZE + ZORDER BY (customer_id, order_date)

  4. STORAGE:
     Bronze: s3://data/bronze/orders/ (Delta, partitioned by date)
     Silver: s3://data/silver/orders/ (Delta, deduped, partitioned by date)
     Gold: s3://data/gold/ (Delta, ZORDER for BI query patterns)
     Dashboard: s3://data/streaming/window_agg/ (Delta, last 24h)

  5. SERVING:
     BI tool (Tableau/Superset) → queries Gold Delta tables
     Dashboard → queries streaming window_agg Delta table
     Athena → ad-hoc queries on Bronze/Silver via Glue Catalog

  6. OBSERVABILITY:
     Streaming: log batch_id, input_rows, rejected_rows, window lag
     Alert: rejection rate > 5%, lag > 5 minutes, batch duration > 90 min
     Spark UI: embedded in EMR/Databricks, check for spill + skew

Trade-offs:
  Kafka vs Kinesis: Kafka more flexible (partitions, compaction), Kinesis simpler on AWS
  EMR vs Databricks: EMR cheaper for steady workload, Databricks for team productivity
  Delta vs Parquet: Delta adds ACID + time travel (worth it for Gold/Silver)
════════════════════════════════════════════════════════════════
""")

## Part 4: Common Mistakes & How to Avoid Them

In [ ]:
print("""
Common Spark Mistakes (and fixes):
════════════════════════════════════════════════════════════════

❌  Using collect() on large DataFrames
    → OOM on driver. Use write() or take(N) for inspection.

❌  groupByKey() instead of reduceByKey() on RDDs
    → groupByKey shuffles ALL values. reduceByKey reduces on each partition first.
    On DataFrames: use groupBy().agg() which is already optimized.

❌  Forgetting unpersist() after cache()
    → Cached DataFrames hold memory until the SparkSession ends or evicted.
    Always: df.unpersist() when you no longer need it.

❌  Using inferSchema in production
    → Reads entire file to infer → slow. Schema can change → breaks pipeline.
    Always: provide explicit StructType.

❌  Accumulator read outside of action
    → Accumulators only update after a completed action.
    Read accumulator.value AFTER df.count() or df.write().

❌  Stream-stream join without watermark
    → State grows indefinitely → OOM.
    Always: withWatermark() on BOTH sides of a stream-stream join.

❌  partitionBy(high_cardinality_column)
    → 1M distinct customer_ids → 1M directories → tiny files everywhere.
    Use low-cardinality columns: date, status, region, category.

❌  Using Python UDFs for math operations
    → 10-100× slower than built-in functions.
    Use F.when(), F.regexp_extract(), F.to_date(), etc.

❌  Chained withColumn for 50 columns
    → Each withColumn creates a new query plan node → very slow plan compilation.
    Use select([...]) with all columns at once.

❌  Joining without statistics
    → Catalyst picks the wrong join algorithm.
    Run ANALYZE TABLE or use hints (broadcast, merge).

❌  Streaming without checkpointLocation
    → Restart = reprocess everything from the beginning.
    Always set .option("checkpointLocation", "s3://...").

❌  Writing to Parquet with many partitions (default 200)
    → 200 files per write. Use coalesce(N) before write.
════════════════════════════════════════════════════════════════
""")

## Practice Challenges — Do These Without Looking!

1. **Join + Window + Rank**: Write a query that returns, for each category, the top 3 customers by total revenue (excluding cancelled orders). Use Window functions with `dense_rank()`.

2. **Deduplication**: Given a `payments` table that may have duplicate `payment_id` (same payment retried twice), keep only the first attempt (lowest `created_at`). Write the code from scratch.

3. **SCD2 MERGE**: Write the two-step SCD2 MERGE for a `customers` table (Step 1: close old rows, Step 2: insert new rows). Assume Delta Lake.

4. **Dead letter pipeline**: Write a function that takes a streaming DataFrame with a JSON `value` column, parses it with `from_json`, and routes valid records to one sink and invalid records (null after parse) to another.

5. **Performance diagnosis**: You have a job with a groupBy that takes 45 minutes. The Spark UI shows one task takes 40 minutes while others finish in 30 seconds. What is the root cause? Write the fix using salting.

6. **System design**: Design a streaming pipeline to compute top-10 trending products per hour from a Kafka stream of click events. Include: source, watermark, window type, output sink, exactly-once strategy.